In [ ]:
#!/usr/bin/env python3

import json
import re
import time
from pathlib import Path
from urllib.parse import urljoin

import requests


# ============================================================
# Configuration
# ============================================================

COMPANIES = {
    "NVIDIA": "0001045810",
    "Microsoft": "0000789019",
    "Apple": "0000320193",
    "Amazon": "0001018724",
    "Alphabet": "0001652044",
    "Meta": "0001326801",
    "Tesla": "0001318605",
    "AMD": "0000002488",
    "Intel": "0000050863",
    "Oracle": "0001341439",
}

FORMS = ["10-K"]

# Number of filings to download per company.
# Start small. Increase later.
FILINGS_PER_COMPANY = 5

BASE_DIR = Path(__file__).resolve().parents[1]
OUTPUT_DIR = BASE_DIR / "data" / "raw" / "sec"

# IMPORTANT:
# Replace this with your own real application/contact information.
USER_AGENT = "KnowledgeOS your-email@example.com"

SEC_DATA_BASE = "https://data.sec.gov"
SEC_ARCHIVE_BASE = "https://www.sec.gov"


# ============================================================
# HTTP
# ============================================================

session = requests.Session()

session.headers.update(
    {
        "User-Agent": USER_AGENT,
        "Accept-Encoding": "gzip, deflate",
        "Host": "data.sec.gov",
    }
)


def get_json(url: str) -> dict:
    """GET JSON from SEC with basic retry handling."""

    for attempt in range(3):
        try:
            response = session.get(url, timeout=30)

            if response.status_code == 429:
                wait = 2 ** attempt
                print(f"Rate limited. Waiting {wait}s...")
                time.sleep(wait)
                continue

            response.raise_for_status()
            return response.json()

        except requests.RequestException as exc:
            if attempt == 2:
                raise

            wait = 2 ** attempt
            print(f"Request failed: {exc}. Retrying in {wait}s...")
            time.sleep(wait)

    raise RuntimeError(f"Unable to fetch {url}")


def download_file(url: str, destination: Path):
    """Download a filing document."""

    headers = {
        "User-Agent": USER_AGENT,
        "Accept-Encoding": "gzip, deflate",
        "Host": "www.sec.gov",
    }

    for attempt in range(3):
        try:
            response = requests.get(
                url,
                headers=headers,
                timeout=60,
            )

            if response.status_code == 429:
                wait = 2 ** attempt
                print(f"Rate limited. Waiting {wait}s...")
                time.sleep(wait)
                continue

            response.raise_for_status()

            destination.write_bytes(response.content)
            return

        except requests.RequestException:
            if attempt == 2:
                raise

            wait = 2 ** attempt
            time.sleep(wait)


# ============================================================
# SEC metadata
# ============================================================

def get_company_submissions(cik: str) -> dict:
    """Get filing history for a company."""

    cik_padded = cik.zfill(10)

    url = (
        f"{SEC_DATA_BASE}/submissions/"
        f"CIK{cik_padded}.json"
    )

    print(f"Fetching submissions: {url}")

    return get_json(url)


def get_recent_filings(submissions: dict):
    """
    Extract recent filings from the SEC compact filing structure.
    """

    recent = submissions["filings"]["recent"]

    filings = []

    for i in range(len(recent["form"])):

        filing = {
            "accession_number": recent["accessionNumber"][i],
            "filing_date": recent["filingDate"][i],
            "report_date": recent["reportDate"][i],
            "form": recent["form"][i],
            "primary_document": recent["primaryDocument"][i],
            "primary_doc_description": recent.get(
                "primaryDocDescription",
                [None] * len(recent["form"]),
            )[i],
            "file_number": recent["fileNumber"][i],
            "items": recent.get(
                "items",
                [None] * len(recent["form"]),
            )[i],
            "size": recent.get(
                "size",
                [None] * len(recent["form"]),
            )[i],
        }

        filings.append(filing)

    return filings


# ============================================================
# URL construction
# ============================================================

def build_filing_url(cik: str, accession_number: str, primary_document: str):
    """
    Build the URL for the primary filing document.

    Example:

    CIK:
        1045810

    Accession:
        0001045810-26-000012

    Archive:
        https://www.sec.gov/Archives/edgar/data/
        1045810/
        000104581026000012/
        primary-document.htm
    """

    cik_numeric = str(int(cik))
    accession_no_dashes = accession_number.replace("-", "")

    return (
        f"{SEC_ARCHIVE_BASE}/Archives/edgar/data/"
        f"{cik_numeric}/"
        f"{accession_no_dashes}/"
        f"{primary_document}"
    )


def build_submission_index_url(
    cik: str,
    accession_number: str,
):
    """
    Build the SEC filing index page.
    """

    cik_numeric = str(int(cik))
    accession_no_dashes = accession_number.replace("-", "")

    return (
        f"{SEC_ARCHIVE_BASE}/Archives/edgar/data/"
        f"{cik_numeric}/"
        f"{accession_no_dashes}/"
        f"{accession_number}-index.html"
    )


# ============================================================
# File utilities
# ============================================================

def safe_filename(value: str) -> str:
    """Make a string safe for filesystem usage."""

    return re.sub(r"[^a-zA-Z0-9._-]", "_", value)


def save_metadata(
    company_dir: Path,
    filing: dict,
    filing_url: str,
    index_url: str,
):
    metadata = {
        "company": filing["company"],
        "ticker": filing.get("ticker"),
        "cik": filing["cik"],
        "form": filing["form"],
        "filing_date": filing["filing_date"],
        "report_date": filing["report_date"],
        "accession_number": filing["accession_number"],
        "primary_document": filing["primary_document"],
        "primary_doc_description": filing[
            "primary_doc_description"
        ],
        "file_number": filing["file_number"],
        "items": filing["items"],
        "size": filing["size"],
        "filing_url": filing_url,
        "index_url": index_url,
        "source": "SEC EDGAR",
    }

    metadata_path = (
        company_dir
        / f"{filing['accession_number']}.json"
    )

    metadata_path.write_text(
        json.dumps(metadata, indent=2),
        encoding="utf-8",
    )


# ============================================================
# Company ingestion
# ============================================================

def ingest_company(company_name: str, cik: str):

    print()
    print("=" * 70)
    print(f"Company: {company_name}")
    print(f"CIK:     {cik}")
    print("=" * 70)

    submissions = get_company_submissions(cik)

    official_name = submissions.get("name", company_name)

    tickers = submissions.get("tickers", [])

    ticker = tickers[0] if tickers else None

    filings = get_recent_filings(submissions)

    selected = [
        filing
        for filing in filings
        if filing["form"] in FORMS
    ]

    selected = selected[:FILINGS_PER_COMPANY]

    if not selected:
        print("No matching filings found.")
        return

    company_dir = (
        OUTPUT_DIR
        / safe_filename(company_name)
        / "10-K"
    )

    company_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    for filing in selected:

        filing["company"] = official_name
        filing["ticker"] = ticker
        filing["cik"] = cik

        accession = filing["accession_number"]
        primary_document = filing["primary_document"]

        filing_url = build_filing_url(
            cik,
            accession,
            primary_document,
        )

        index_url = build_submission_index_url(
            cik,
            accession,
        )

        filename = (
            f"{filing['filing_date']}_"
            f"{accession}_"
            f"{safe_filename(primary_document)}"
        )

        destination = company_dir / filename

        metadata_path = (
            company_dir
            / f"{accession}.json"
        )

        print()
        print(f"Form:     {filing['form']}")
        print(f"Filed:    {filing['filing_date']}")
        print(f"Report:   {filing['report_date']}")
        print(f"Accession:{accession}")
        print(f"Document: {primary_document}")

        if destination.exists():
            print("Already downloaded. Skipping.")
        else:
            print(f"Downloading: {filing_url}")

            download_file(
                filing_url,
                destination,
            )

            print(f"Saved: {destination}")

            # Be polite to SEC servers.
            time.sleep(0.2)

        if not metadata_path.exists():

            save_metadata(
                company_dir,
                filing,
                filing_url,
                index_url,
            )

        time.sleep(0.2)


# ============================================================
# Main
# ============================================================

def main():

    OUTPUT_DIR.mkdir(
        parents=True,
        exist_ok=True,
    )

    print("KnowledgeOS - SEC EDGAR ingestion")
    print(f"Output: {OUTPUT_DIR}")
    print(f"Forms: {FORMS}")
    print(
        f"Filings/company: "
        f"{FILINGS_PER_COMPANY}"
    )

    for company_name, cik in COMPANIES.items():

        try:
            ingest_company(
                company_name,
                cik,
            )

        except Exception as exc:
            print(
                f"\nERROR processing "
                f"{company_name}: {exc}"
            )

        # Keep requests conservative.
        time.sleep(1)

    print()
    print("=" * 70)
    print("SEC ingestion completed.")
    print("=" * 70)


if __name__ == "__main__":
    main()

In [ ]:
#!/usr/bin/env python3

import json
import re
import time
from pathlib import Path
from urllib.parse import urljoin

import requests


# ============================================================
# Configuration
# ============================================================

COMPANIES = {
    "NVIDIA": "0001045810",
    "Microsoft": "0000789019",
    "Apple": "0000320193",
    "Amazon": "0001018724",
    "Alphabet": "0001652044",
    "Meta": "0001326801",
    "Tesla": "0001318605",
    "AMD": "0000002488",
    "Intel": "0000050863",
    "Oracle": "0001341439",
}

FORMS = ["10-K"]

# Number of filings to download per company.
# Start small. Increase later.
FILINGS_PER_COMPANY = 5

BASE_DIR = Path(__file__).resolve().parents[1]
OUTPUT_DIR = BASE_DIR / "data" / "raw" / "sec"

# IMPORTANT:
# Replace this with your own real application/contact information.
USER_AGENT = "KnowledgeOS your-email@example.com"

SEC_DATA_BASE = "https://data.sec.gov"
SEC_ARCHIVE_BASE = "https://www.sec.gov"


# ============================================================
# HTTP
# ============================================================

session = requests.Session()

session.headers.update(
    {
        "User-Agent": USER_AGENT,
        "Accept-Encoding": "gzip, deflate",
        "Host": "data.sec.gov",
    }
)


def get_json(url: str) -> dict:
    """GET JSON from SEC with basic retry handling."""

    for attempt in range(3):
        try:
            response = session.get(url, timeout=30)

            if response.status_code == 429:
                wait = 2 ** attempt
                print(f"Rate limited. Waiting {wait}s...")
                time.sleep(wait)
                continue

            response.raise_for_status()
            return response.json()

        except requests.RequestException as exc:
            if attempt == 2:
                raise

            wait = 2 ** attempt
            print(f"Request failed: {exc}. Retrying in {wait}s...")
            time.sleep(wait)

    raise RuntimeError(f"Unable to fetch {url}")


def download_file(url: str, destination: Path):
    """Download a filing document."""

    headers = {
        "User-Agent": USER_AGENT,
        "Accept-Encoding": "gzip, deflate",
        "Host": "www.sec.gov",
    }

    for attempt in range(3):
        try:
            response = requests.get(
                url,
                headers=headers,
                timeout=60,
            )

            if response.status_code == 429:
                wait = 2 ** attempt
                print(f"Rate limited. Waiting {wait}s...")
                time.sleep(wait)
                continue

            response.raise_for_status()

            destination.write_bytes(response.content)
            return

        except requests.RequestException:
            if attempt == 2:
                raise

            wait = 2 ** attempt
            time.sleep(wait)


# ============================================================
# SEC metadata
# ============================================================

def get_company_submissions(cik: str) -> dict:
    """Get filing history for a company."""

    cik_padded = cik.zfill(10)

    url = (
        f"{SEC_DATA_BASE}/submissions/"
        f"CIK{cik_padded}.json"
    )

    print(f"Fetching submissions: {url}")

    return get_json(url)


def get_recent_filings(submissions: dict):
    """
    Extract recent filings from the SEC compact filing structure.
    """

    recent = submissions["filings"]["recent"]

    filings = []

    for i in range(len(recent["form"])):

        filing = {
            "accession_number": recent["accessionNumber"][i],
            "filing_date": recent["filingDate"][i],
            "report_date": recent["reportDate"][i],
            "form": recent["form"][i],
            "primary_document": recent["primaryDocument"][i],
            "primary_doc_description": recent.get(
                "primaryDocDescription",
                [None] * len(recent["form"]),
            )[i],
            "file_number": recent["fileNumber"][i],
            "items": recent.get(
                "items",
                [None] * len(recent["form"]),
            )[i],
            "size": recent.get(
                "size",
                [None] * len(recent["form"]),
            )[i],
        }

        filings.append(filing)

    return filings


# ============================================================
# URL construction
# ============================================================

def build_filing_url(cik: str, accession_number: str, primary_document: str):
    """
    Build the URL for the primary filing document.

    Example:

    CIK:
        1045810

    Accession:
        0001045810-26-000012

    Archive:
        https://www.sec.gov/Archives/edgar/data/
        1045810/
        000104581026000012/
        primary-document.htm
    """

    cik_numeric = str(int(cik))
    accession_no_dashes = accession_number.replace("-", "")

    return (
        f"{SEC_ARCHIVE_BASE}/Archives/edgar/data/"
        f"{cik_numeric}/"
        f"{accession_no_dashes}/"
        f"{primary_document}"
    )


def build_submission_index_url(
    cik: str,
    accession_number: str,
):
    """
    Build the SEC filing index page.
    """

    cik_numeric = str(int(cik))
    accession_no_dashes = accession_number.replace("-", "")

    return (
        f"{SEC_ARCHIVE_BASE}/Archives/edgar/data/"
        f"{cik_numeric}/"
        f"{accession_no_dashes}/"
        f"{accession_number}-index.html"
    )


# ============================================================
# File utilities
# ============================================================

def safe_filename(value: str) -> str:
    """Make a string safe for filesystem usage."""

    return re.sub(r"[^a-zA-Z0-9._-]", "_", value)


def save_metadata(
    company_dir: Path,
    filing: dict,
    filing_url: str,
    index_url: str,
):
    metadata = {
        "company": filing["company"],
        "ticker": filing.get("ticker"),
        "cik": filing["cik"],
        "form": filing["form"],
        "filing_date": filing["filing_date"],
        "report_date": filing["report_date"],
        "accession_number": filing["accession_number"],
        "primary_document": filing["primary_document"],
        "primary_doc_description": filing[
            "primary_doc_description"
        ],
        "file_number": filing["file_number"],
        "items": filing["items"],
        "size": filing["size"],
        "filing_url": filing_url,
        "index_url": index_url,
        "source": "SEC EDGAR",
    }

    metadata_path = (
        company_dir
        / f"{filing['accession_number']}.json"
    )

    metadata_path.write_text(
        json.dumps(metadata, indent=2),
        encoding="utf-8",
    )


# ============================================================
# Company ingestion
# ============================================================

def ingest_company(company_name: str, cik: str):

    print()
    print("=" * 70)
    print(f"Company: {company_name}")
    print(f"CIK:     {cik}")
    print("=" * 70)

    submissions = get_company_submissions(cik)

    official_name = submissions.get("name", company_name)

    tickers = submissions.get("tickers", [])

    ticker = tickers[0] if tickers else None

    filings = get_recent_filings(submissions)

    selected = [
        filing
        for filing in filings
        if filing["form"] in FORMS
    ]

    selected = selected[:FILINGS_PER_COMPANY]

    if not selected:
        print("No matching filings found.")
        return

    company_dir = (
        OUTPUT_DIR
        / safe_filename(company_name)
        / "10-K"
    )

    company_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    for filing in selected:

        filing["company"] = official_name
        filing["ticker"] = ticker
        filing["cik"] = cik

        accession = filing["accession_number"]
        primary_document = filing["primary_document"]

        filing_url = build_filing_url(
            cik,
            accession,
            primary_document,
        )

        index_url = build_submission_index_url(
            cik,
            accession,
        )

        filename = (
            f"{filing['filing_date']}_"
            f"{accession}_"
            f"{safe_filename(primary_document)}"
        )

        destination = company_dir / filename

        metadata_path = (
            company_dir
            / f"{accession}.json"
        )

        print()
        print(f"Form:     {filing['form']}")
        print(f"Filed:    {filing['filing_date']}")
        print(f"Report:   {filing['report_date']}")
        print(f"Accession:{accession}")
        print(f"Document: {primary_document}")

        if destination.exists():
            print("Already downloaded. Skipping.")
        else:
            print(f"Downloading: {filing_url}")

            download_file(
                filing_url,
                destination,
            )

            print(f"Saved: {destination}")

            # Be polite to SEC servers.
            time.sleep(0.2)

        if not metadata_path.exists():

            save_metadata(
                company_dir,
                filing,
                filing_url,
                index_url,
            )

        time.sleep(0.2)


# ============================================================
# Main
# ============================================================

def main():

    OUTPUT_DIR.mkdir(
        parents=True,
        exist_ok=True,
    )

    print("KnowledgeOS - SEC EDGAR ingestion")
    print(f"Output: {OUTPUT_DIR}")
    print(f"Forms: {FORMS}")
    print(
        f"Filings/company: "
        f"{FILINGS_PER_COMPANY}"
    )

    for company_name, cik in COMPANIES.items():

        try:
            ingest_company(
                company_name,
                cik,
            )

        except Exception as exc:
            print(
                f"\nERROR processing "
                f"{company_name}: {exc}"
            )

        # Keep requests conservative.
        time.sleep(1)

    print()
    print("=" * 70)
    print("SEC ingestion completed.")
    print("=" * 70)


if __name__ == "__main__":
    main()